# PROJECT : Trading Game #1 — Multivariate Statistical Arbitrage

Commodity: Sugar (G18)

**Strategy**: Construct an optimal market-neutral basket of equities cointegrated with Sugar Futures  
**Features**: Engle-Granger cointegration selection · OLS hedge ratios · Gurobi portfolio optimization  
**Metrics**: Sharpe Ratio · Max Drawdown · Calmar Ratio

---

In [ ]:
!pip install gurobipy
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
from gurobipy import Model, GRB

Here upload the dataset : **sugar_dataset_G18**

In [ ]:
ANCHOR = 'SB=F'  # Sugar futures

# ============================================================
# Data Collection — G18 Sugar
# Originally downloaded via yfinance (2019-2024, auto_adjust=True)
# Cleaned: dropped tickers with >20% missing (CSAN, SGG),
# forward-filled gaps, dropped remaining NaNs.
# Final dataset: 18 assets × 1510 trading days
#
# To replicate the download from scratch:
# raw = yf.download(ALL_ASSETS, start='2019-01-01', end='2024-12-31', auto_adjust=True)
# data_full = raw['Close'].dropna(axis=1, thresh=int(len(raw)*0.8)).ffill().dropna()
# ============================================================

from google.colab import files
uploaded = files.upload()

import io
filename = list(uploaded.keys())[0]
raw_csv = pd.read_csv(io.BytesIO(uploaded[filename]), index_col=0, parse_dates=True)

# Keep only Close columns and rename to ticker names
close_cols = [col for col in raw_csv.columns if col.endswith('_Close')]
data_full = raw_csv[close_cols].copy()
data_full.columns = [col.replace('_Close', '') for col in data_full.columns]

stocks = [c for c in data_full.columns if c != ANCHOR]

print(f"Period : {data_full.index[0].date()} → {data_full.index[-1].date()}")
print(f"Kept assets: {list(data_full.columns)}")
print(data_full.head())

In [ ]:
data_log = np.log(data_full)

pvalues = {}
for stock in stocks:
    score, pvalue, _ = coint(data_log[ANCHOR], data_log[stock])
    pvalues[stock] = pvalue

selected_stocks = sorted(pvalues, key=pvalues.get)[:5]
print("Selected stocks:", selected_stocks)
print("p-values:", {s: round(pvalues[s], 4) for s in selected_stocks})

In [ ]:
X = data_log[selected_stocks]
X = sm.add_constant(X)
y = data_log[ANCHOR]

model = sm.OLS(y, X).fit()
betas = model.params[1:]
print("OLS Betas:")
print(betas)

In [ ]:
returns = data_full[selected_stocks].pct_change().dropna()
cov_matrix = returns.cov().values
n = len(selected_stocks)

m = Model()
m.setParam('OutputFlag', 0)

w = m.addVars(n, lb=-1, ub=1, name='w')

m.setObjective(
    sum(w[i] * cov_matrix[i, j] * w[j]
        for i in range(n) for j in range(n)),
    GRB.MINIMIZE
)

# ∑β·w = 0
m.addConstr(
    sum(w[i] * float(betas.iloc[i]) for i in range(n)) == 0
)

# ∑w = 1
m.addConstr(sum(w[i] for i in range(n)) == 1)


m.optimize()
print("Status Gurobi:", m.status, "(2 = OPTIMAL)")

if m.status == GRB.OPTIMAL:
    opt_weights = np.array([w[i].x for i in range(n)])
    print("optimal weights :", dict(zip(selected_stocks, opt_weights.round(4))))
    print("∑w =", round(opt_weights.sum(), 6))
    print("∑β·w =", round(sum(opt_weights[i] * float(betas.iloc[i]) for i in range(n)), 6))
else:
    print("⚠️ Infeasible — fallback poids égaux")
    opt_weights = np.ones(n) / n


In [ ]:
basket = (data_log[selected_stocks] * opt_weights).sum(axis=1)
spread = data_log[ANCHOR] - basket

window = 30
zscore = (spread - spread.rolling(window).mean()) / spread.rolling(window).std()

# Test ADF for stationarity
from statsmodels.tsa.stattools import adfuller
adf = adfuller(spread.dropna())
print(f"\nADF p-value : {adf[1]:.4f} → {'stable' if adf[1] < 0.05 else ' non stable'}")

plt.figure()
spread.plot(title='Spread (log-prix)')
plt.show()

plt.figure()
zscore.plot(title='Z-Score')
plt.axhline( 2, color='r', linestyle='--')
plt.axhline(-2, color='r', linestyle='--')
plt.show()

In [ ]:
signals = pd.Series(0, index=spread.index)
signals[zscore >  2] = -1
signals[zscore < -2] =  1

signals[zscore.abs() < 0.5] = 0

signals = signals.fillna(0)

print(signals.value_counts())

In [ ]:
returns_anchor = data_full[ANCHOR].pct_change()
returns_basket = sum(
    opt_weights[i] * data_full[selected_stocks[i]].pct_change()
    for i in range(len(selected_stocks))
)

strategy_returns = signals.shift(1) * (returns_anchor - returns_basket)
tc = 0.001
strategy_returns -= tc * signals.diff().abs().fillna(0)
cum_returns = (1 + strategy_returns).cumprod()

plt.figure()
cum_returns.plot(title='Strategy Performance')
plt.show()

In [ ]:
def sharpe_ratio(returns):
    return np.sqrt(252) * returns.mean() / returns.std()


def max_drawdown(cum_returns):
    peak = cum_returns.cummax()
    drawdown = (cum_returns - peak) / peak
    return drawdown.min()

sharpe = sharpe_ratio(strategy_returns.dropna())
mdd = max_drawdown(cum_returns)
calmar = (cum_returns.iloc[-1] - 1) / abs(mdd)

print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Max Drawdown: {mdd:.2%}")
print(f"Calmar Ratio: {calmar:.2f}")


In [ ]:
benchmark = data_full[ANCHOR].pct_change()
benchmark_cum = (1 + benchmark).cumprod()

plt.figure()
cum_returns.plot(label='Strategy')
benchmark_cum.plot(label='Benchmark (Sugar)')
plt.legend()
plt.title('Strategy vs Benchmark')
plt.show()
